Basic XGB Pipeline

In [13]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

import warnings
warnings.filterwarnings('ignore')

In [14]:
train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")
test["accident_risk"] = 0.5  # Initialize with placeholder value

In [15]:
# Load synthetic road accident datasets (multiple sizes)
synthetic = []
for size in [2, 10, 100]:
    df = pd.read_csv(f"synthetic_data/synthetic_road_accidents_{size}k.csv")
    synthetic.append(df)
synthetic = pd.concat(synthetic, axis=0)

# Align columns and add new IDs for synthetic data
synthetic["id"] = np.arange(len(synthetic)) + test["id"].max() + 1
synthetic = synthetic[train.columns]

# Combine all data for unified preprocessing
combined = pd.concat([train, test, synthetic], axis=0, ignore_index=True)

In [16]:
# Feature Engineering

FEATURES = list(synthetic.columns[1:-1])
TARGET = synthetic.columns[-1]

# Custom feature 'y' based on road & weather conditions
def road_risk(X):
    return (
        0.3 * X["curvature"] +
        0.2 * (X["lighting"] == "night").astype(int) +
        0.1 * (X["weather"] != "clear").astype(int) +
        0.2 * (X["speed_limit"] >= 60).astype(int) +
        0.1 * (X["num_reported_accidents"] > 2).astype(int)
    )

# Smoothed clipping using normal distribution
def clipped(func):
    def clip_f(X):
        mu = func(X)
        sigma = 0.05
        a, b = -mu / sigma, (1 - mu) / sigma
        Phi_a, Phi_b = stats.norm.cdf(a), stats.norm.cdf(b)
        phi_a, phi_b = stats.norm.pdf(a), stats.norm.pdf(b)
        return mu * (Phi_b - Phi_a) + sigma * (phi_a - phi_b) + 1 - Phi_b
    return clip_f

combined["y"] = clipped(road_risk)(combined)

# Add feature interaction: curvature * speed_limit
combined["curv_speed_interaction"] = combined["curvature"] * combined["speed_limit"]
FEATURES.append("y")
FEATURES.append("curv_speed_interaction")

In [17]:
# Handle Categorical Data

CATS, NUMS = [], []
for col in FEATURES:
    if combined[col].dtype == "object":
        CATS.append(col)
    else:
        NUMS.append(col)

# Factorize (encode) categorical columns
for col in CATS:
    combined[col], _ = combined[col].factorize()
    combined[col] = combined[col].astype("int32")

In [18]:
# Split Data Back

train = combined.iloc[:len(train)]
test = combined.iloc[len(train):len(train) + len(test)]
synthetic = combined.iloc[-len(synthetic):]

In [19]:
# Target Encoding

TE_features = []
for col in FEATURES:
    te_map = synthetic.groupby(col)[TARGET].mean()
    te_name = f"TE_{col}"
    train = train.merge(te_map.rename(te_name), on=col, how="left")
    test = test.merge(te_map.rename(te_name), on=col, how="left")
    TE_features.append(te_name)

In [ ]:
# Model Training (XGBoost)

params = {
    "objective": "reg:squarederror",
    "eval_metric": "rmse",
    "learning_rate": 0.01,
    "max_depth": 6,
    "subsample": 0.9,
    "colsample_bytree": 0.6,
    "seed": 42,
    "device": "cuda",
}

FOLDS = 7
kf = KFold(n_splits=FOLDS, shuffle=True, random_state=2025)

oof_preds = np.zeros(len(train))
test_preds = np.zeros(len(test))

print("\n Training model with 7-Fold Cross Validation...\n")

for fold, (train_idx, val_idx) in enumerate(kf.split(train)):
    print(f" Fold {fold+1}/{FOLDS}")
    
    X_train = train.iloc[train_idx][FEATURES + TE_features]
    y_train = train.iloc[train_idx][TARGET] - train.iloc[train_idx]["y"]

    X_val = train.iloc[val_idx][FEATURES + TE_features]
    y_val = train.iloc[val_idx][TARGET] - train.iloc[val_idx]["y"]
    y_val_base = train.iloc[val_idx]["y"].values

    dtrain = xgb.DMatrix(X_train, label=y_train)
    dval = xgb.DMatrix(X_val, label=y_val)
    dtest = xgb.DMatrix(test[FEATURES + TE_features])

    model = xgb.train(
        params=params,
        dtrain=dtrain,
        num_boost_round=100_000,
        evals=[(dtrain, "Train"), (dval, "Valid")],
        early_stopping_rounds=200,
        verbose_eval=False
    )

    oof_preds[val_idx] = model.predict(dval) + y_val_base
    test_preds += (model.predict(dtest) + test["y"].values) / FOLDS


🚀 Training model with 7-Fold Cross Validation...

📂 Fold 1/7
📂 Fold 2/7
📂 Fold 3/7
📂 Fold 4/7
📂 Fold 5/7
📂 Fold 6/7
📂 Fold 7/7


In [21]:
# Evaluate Model

rmse_model = np.sqrt(mean_squared_error(train[TARGET], oof_preds))
rmse_baseline = np.sqrt(mean_squared_error(train[TARGET], train["y"]))

print(f"\n✅ Model RMSE: {rmse_model:.5f}")
print(f"📉 Baseline RMSE: {rmse_baseline:.5f}\n")


✅ Model RMSE: 0.05596
📉 Baseline RMSE: 0.05854



In [22]:
# Submission

sub = pd.read_csv("data/sample_submission.csv")
sub["accident_risk"] = test_preds
sub.to_csv("submission.csv", index=False)

print("\n📤 Submission File Preview:")
sub.head()


📤 Submission File Preview:


,id,accident_risk
0,517754,0.294817
1,517755,0.119557
2,517756,0.180189
3,517757,0.307316
4,517758,0.393059
